# 🛡️ TypedDict vs Pydantic State in LangGraph

## Learning Objectives
In this notebook, you will learn:
1. **TypedDict State** — The default LangGraph state schema; type hints with no runtime enforcement
2. **Pydantic BaseModel State** — Drop-in replacement that adds automatic runtime type validation
3. **Key Differences** — When to choose each and what breaks when input is invalid
4. **Annotated Reducers** — How `add_messages` works identically in both schemas

## Prerequisites
- LangGraph basics: `01_State_and_Graph_Basics.ipynb`
- Python type hints and `typing` module
- Basic familiarity with Pydantic `BaseModel`

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: Imports
# ============================================================================
from typing import Annotated, TypedDict

from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

print("✅ Imports loaded")

✅ Imports loaded


---
## 📋 Part 1: TypedDict State — Type Hints Without Enforcement

`TypedDict` is the default way to define LangGraph state. It uses standard
Python type annotations, which serve as **documentation and IDE hints only** —
they are **not enforced at runtime**. LangGraph passes whatever dict values you
provide directly into your nodes, regardless of type.

### Key Concepts:
- **Structural typing** — `TypedDict` declares the expected *shape* of the state dict
- **No runtime validation** — passing an `int` where `str` is annotated will not raise any error
- **Lightweight** — minimal overhead; ideal when you control all inputs

In [ ]:
# ============================================================================
# STATE SCHEMA: TypedDict with a plain str field
# ============================================================================
class TypedDictState(TypedDict):
    name: str  # annotation only — NOT enforced at runtime


def respond(state: TypedDictState) -> dict:
    name = state["name"]
    return {"name":name + " How are you?"}


# ---
# GRAPH ASSEMBLY: START → greet → respond → END
# ---
builder = StateGraph(TypedDictState)

builder.add_node("respond", respond)

builder.add_edge(START, "respond")

builder.add_edge("respond", END)

typeddict_graph = builder.compile()
print("✅ TypedDictState graph compiled")

✅ TypedDictState graph compiled


### ✅ Valid Input — Works as Expected

Passing a string flows through both nodes. Each node overwrites `name`,
so the final state holds the value from the last node (`respond`).

In [28]:
# ============================================================================
# VALID INVOCATION: str input flows normally through both nodes
# ============================================================================
result = typeddict_graph.invoke({"name": "Sourav"})
print(f"🧮 Final state: {result}")

🧮 Final state: {'name': 'Sourav How are you?'}


### ⚠️ Invalid Input — No Error Raised!

Passing an `int` where `str` is annotated **silently succeeds** with `TypedDict`.
The graph runs, the int is passed into the `greet` node, and Python raises a
`TypeError` only when the node tries to do string formatting — *deep inside*
your logic, not at the boundary.

> **Note:** This is the core risk of `TypedDict` — bad inputs can travel deep
> into your graph before you discover there's a problem.

In [29]:
# ============================================================================
# INVALID INVOCATION: int input — TypedDict does NOT catch this at graph entry
# ============================================================================
try:
    result = typeddict_graph.invoke({"name": 123})
    print(f"🧮 Final state: {result}")
except Exception as e:
    # TypeError fires INSIDE the greet node during f-string formatting,
    # not at graph.invoke() — the bad value slipped past the boundary
    print(f"⚠️  Caught {type(e).__name__} (inside node, not at graph entry):")
    print(f"   {e}")

⚠️  Caught TypeError (inside node, not at graph entry):
   unsupported operand type(s) for +: 'int' and 'str'


---
## 🛡️ Part 2: Pydantic BaseModel State — Runtime Validation

`BaseModel` is a drop-in replacement for `TypedDict` in LangGraph. The graph
wiring, nodes, and edges are identical — the only change is the state class
declaration. But with Pydantic, every field type **is enforced at runtime**:
LangGraph validates the input dict against the schema *before* calling any node.

### Key Concepts:
- **Fail-fast** — `ValidationError` is raised at `graph.invoke()`, not buried in node logic
- **Immutable in nodes** — Pydantic models are immutable; nodes must *return* update dicts, never mutate `state` attributes
- **Rich validation** — supports default values, `@field_validator`, custom constraints, and more

In [39]:
# ============================================================================
# STATE SCHEMA: Pydantic BaseModel — same field, now enforced at runtime
# ============================================================================
class PydanticState(BaseModel):
    name: str  # Pydantic WILL reject any non-string value at graph.invoke()


# ---
# GRAPH NODES: Access state via attribute syntax (state.name, not state['name'])
# ---
def greet_p(state: PydanticState) -> dict:
    return {"name": f"Hello, {state.name}!"}

def respond_p(state: PydanticState) -> dict:
    return {"name": "How are you?"}


# ---
# GRAPH ASSEMBLY: identical structure to the TypedDict graph
# ---
builder_p = StateGraph(PydanticState)
builder_p.add_node("greet_p", greet_p)
builder_p.add_node("respond_p", respond_p)

builder_p.add_edge(START, "greet_p")
builder_p.add_edge("greet_p", "respond_p")
builder_p.add_edge("respond_p", END)

pydantic_graph = builder_p.compile()
print("✅ PydanticState graph compiled")

✅ PydanticState graph compiled


✅ PydanticState graph compiled


### ✅ Valid Input — Works as Expected

In [40]:
# ============================================================================
# VALID INVOCATION: str input passes Pydantic validation and runs both nodes
# ============================================================================
result = pydantic_graph.invoke({"name": "Sourav"})
print(f"🧮 Final state: {result}")

🧮 Final state: {'name': 'Sourav How are you?'}


### ❌ Invalid Input — ValidationError at Graph Entry

The same `int` input that silently passed in TypedDict now raises a
`ValidationError` **immediately at `graph.invoke()`** — before a single
node executes. The error message tells you exactly which field failed and why.

In [41]:
# ============================================================================
# INVALID INVOCATION: int input is caught at graph.invoke(), no node runs
# ============================================================================
try:
    pydantic_graph.invoke({"name": 123})
except Exception as e:
    print(f"❌ Caught {type(e).__name__} (at graph entry — no node ran):")
    print(e)

❌ Caught ValidationError (at graph entry — no node ran):
1 validation error for PydanticState
name
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type


---
## ⚖️ Part 3: Head-to-Head Comparison

The table below summarises when to choose each approach.

| Feature | `TypedDict` | Pydantic `BaseModel` |
|---|---|---|
| Runtime type validation | ❌ No — hints only | ✅ Yes — enforced at `invoke()` |
| Where errors surface | Deep in node logic | At graph entry, before any node |
| State access in nodes | `state["key"]` (dict-style) | `state.key` (attribute-style) |
| Default field values | Needs `Optional` + `None` | Built-in `= default` |
| Custom validators | ❌ No | ✅ `@field_validator`, constraints |
| Annotated reducers | ✅ Yes | ✅ Yes (identical syntax) |
| Performance overhead | Lower | Slightly higher (parsing cost) |
| Best for | Internal graphs, prototypes | Externally-facing graphs, production |

> **Rule of thumb:** Use `TypedDict` when you fully control the inputs (e.g., internal
> pipelines, notebooks). Switch to `BaseModel` when inputs arrive from users,
> APIs, or external systems where bad data is possible.

---
## 📨 Part 4: Annotated Reducers — Same Syntax in Both Schemas

`Annotated` reducers like `add_messages` work identically in both `TypedDict`
and `BaseModel`. Instead of last-write-wins, the reducer **accumulates** values
across every node that writes to the field.

### Key Concepts:
- **`Annotated[list, add_messages]`** — the reducer function is applied on every state update
- **String coercion** — plain strings passed as values are automatically wrapped in `HumanMessage`
- **Schema-agnostic** — the reducer behaviour is identical regardless of TypedDict vs BaseModel

In [42]:
# ============================================================================
# TYPEDDICT WITH REDUCER: messages accumulate via add_messages
# ============================================================================
class TypedDictMessages(TypedDict):
    messages: Annotated[list, add_messages]


def add_hello(state: TypedDictMessages) -> dict:
    return {"messages": "Hello!"}

def add_reply(state: TypedDictMessages) -> dict:
    return {"messages": "How are you?"}


td_msg_builder = StateGraph(TypedDictMessages)
td_msg_builder.add_node("add_hello", add_hello)
td_msg_builder.add_node("add_reply", add_reply)
td_msg_builder.add_edge(START, "add_hello")
td_msg_builder.add_edge("add_hello", "add_reply")
td_msg_builder.add_edge("add_reply", END)
td_msg_graph = td_msg_builder.compile()

result = td_msg_graph.invoke({"messages": "Sourav"})

print("📨 TypedDict — accumulated messages:")
for msg in result["messages"]:
    print(f"  - {type(msg).__name__}: {msg.content!r}")

📨 TypedDict — accumulated messages:
  - HumanMessage: 'Sourav'
  - HumanMessage: 'Hello!'
  - HumanMessage: 'How are you?'


In [43]:
# ============================================================================
# PYDANTIC BASEMODEL WITH REDUCER: identical reducer behaviour
# ============================================================================
class PydanticMessages(BaseModel):
    messages: Annotated[list, add_messages]


def add_hello_p(state: PydanticMessages) -> dict:
    return {"messages": "Hello!"}

def add_reply_p(state: PydanticMessages) -> dict:
    return {"messages": "How are you?"}


p_msg_builder = StateGraph(PydanticMessages)
p_msg_builder.add_node("add_hello_p", add_hello_p)
p_msg_builder.add_node("add_reply_p", add_reply_p)
p_msg_builder.add_edge(START, "add_hello_p")
p_msg_builder.add_edge("add_hello_p", "add_reply_p")
p_msg_builder.add_edge("add_reply_p", END)
p_msg_graph = p_msg_builder.compile()

result = p_msg_graph.invoke({"messages": "Sourav"})

print("📨 Pydantic BaseModel — accumulated messages (identical output):")
for msg in result["messages"]:
    print(f"  - {type(msg).__name__}: {msg.content!r}")

📨 Pydantic BaseModel — accumulated messages (identical output):
  - HumanMessage: 'Sourav'
  - HumanMessage: 'Hello!'
  - HumanMessage: 'How are you?'


---
## 📝 Summary

In this notebook, we compared `TypedDict` and Pydantic `BaseModel` as LangGraph state schemas.

### 1. TypedDict
- Type annotations are **hints only** — no runtime enforcement
- Invalid input silently enters the graph; errors surface *inside* node logic
- State accessed with **dict syntax**: `state["key"]`
- Best for internal graphs and rapid prototyping

### 2. Pydantic BaseModel
- Type annotations are **enforced at `graph.invoke()`** — fails fast at the boundary
- `ValidationError` fires before any node runs, with a clear field-level message
- State accessed with **attribute syntax**: `state.key`
- Nodes must return dicts — never mutate a Pydantic state object directly
- Best for production graphs and any system accepting external inputs

### 3. Annotated Reducers
- `Annotated[list, add_messages]` works **identically** in both schemas
- Strings are coerced to `HumanMessage`; values accumulate across all nodes

### Next Steps
- See `04_Augmented_LLM_with_Tools.ipynb` to add tool-calling to your graphs
- Explore `03_Advanced_Agents/02_Memory/` for checkpointer-based persistent state